# UNSW-NB15 — Network Intrusion Flows

| | |
|---|---|
| **Source** | Kaggle `dhoogla/unswnb15` (curated parquet of official UNSW-NB15 train/test split) |
| **Origin** | ACCS, UNSW Canberra — Moustafa & Slay 2015 |
| **License** | CC BY 4.0 (per Kaggle listing) |
| **Samples** | 257,673 flows (175,341 train + 82,332 test) |
| **Features** | 34 numeric/categorical flow features + `attack_cat` + `label` |
| **Labels** | binary `label` + 9 attack categories + Normal |
| **Caveats** | This curated version drops raw IPs/ports/timestamps; capture windows were 2015-01-22 and 2015-02-17. Synthetic timestamps assigned (flagged). |
| **Production research suitability** | HIGH — widely benchmarked, labeled, realistic mixed traffic. Not toy. |

In [2]:
import sys, json
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, iqr_outlier_share, save_clean, save_unified_part

pd.set_option("display.max_columns", 50)
D = RAW / "cyber" / "unsw_nb15"

ModuleNotFoundError: No module named 'prep_utils'

## Load

In [3]:
train = pd.read_parquet(D / "UNSW_NB15_training-set.parquet")
test = pd.read_parquet(D / "UNSW_NB15_testing-set.parquet")
train["split"], test["split"] = "train", "test"
df = pd.concat([train, test], ignore_index=True)
print(df.shape)
df.head(3)

NameError: name 'D' is not defined

## Cleaning — duplicates, missing, dtypes

In [3]:
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "split"]).reset_index(drop=True)
print(f"dropped {before - len(df)} duplicate flows")
print("missing values:", int(df.isna().sum().sum()))

dropped 112451 duplicate flows
missing values: 0


In [4]:
# categorical normalization
for c in ["proto", "service", "state", "attack_cat"]:
    df[c] = df[c].astype(str).str.strip().str.lower().astype("category")
df["service"] = df["service"].cat.rename_categories(lambda s: "unknown" if s == "-" else s)
df["attack_cat"].value_counts()

attack_cat
normal            79965
exploits          26749
fuzzers           18313
reconnaissance     8012
dos                4873
generic            3057
analysis           1442
backdoor           1389
shellcode          1257
worms               165
Name: count, dtype: int64

## Label verification
`label` must equal 1 exactly when `attack_cat != normal`.

In [5]:
mismatch = ((df["attack_cat"] != "normal") != (df["label"] == 1)).sum()
print("label/attack_cat mismatches:", mismatch)
assert mismatch == 0
df["label"].value_counts(normalize=True)

label/attack_cat mismatches: 0


label
0    0.55064
1    0.44936
Name: proportion, dtype: float64

## Timestamp normalization
Curated version has no raw timestamps. Capture ran in two windows
(2015-01-22, 2015-02-17; 16h each per dataset paper). Assign uniform synthetic
times inside those windows, preserving nothing temporal — flag `time_is_synthetic`.

In [6]:
rng = np.random.default_rng(42)
w1 = pd.Timestamp("2015-01-22 00:00:00", tz="UTC")
w2 = pd.Timestamp("2015-02-17 00:00:00", tz="UTC")
half = len(df) // 2
offs = rng.uniform(0, 16 * 3600, len(df))
starts = np.where(np.arange(len(df)) < half, w1.value, w2.value)
df["event_time"] = pd.to_datetime(starts + (offs * 1e9).astype("int64"), utc=True)
df = df.sort_values("event_time").reset_index(drop=True)

## Outlier inspection + numeric normalization
Flow features are heavy-tailed. Keep raw values in clean output (models decide
scaling); record outlier shares + store log1p variants for the worst columns.

In [7]:
num_cols = df.select_dtypes(include=[np.number]).columns.drop(["label"])
out_share = {c: iqr_outlier_share(df[c]) for c in num_cols}
worst = sorted(out_share.items(), key=lambda kv: -kv[1])[:10]
print("worst IQR outlier shares:", worst)
for c in ["sbytes", "dbytes", "sload", "dload", "dur"]:
    df[f"log1p_{c}"] = np.log1p(df[c])

worst IQR outlier shares: [('dmean', 0.2008166806682183), ('dbytes', 0.19991461348831444), ('spkts', 0.16614562531847793), ('dpkts', 0.15612648221343872), ('sload', 0.15544476732175566), ('smean', 0.15021140047651182), ('dloss', 0.14373855201002603), ('dload', 0.13456638801283552), ('djit', 0.13451818594978723), ('sbytes', 0.12148297089972594)]


## EDA

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["attack_cat"].value_counts().plot.barh(ax=axes[0], title="attack categories")
df["proto"].value_counts().head(10).plot.barh(ax=axes[1], title="top protocols")
axes[2].hist(df["log1p_sbytes"], bins=60); axes[2].set_title("log1p(sbytes)")
plt.tight_layout(); plt.savefig("../reports/unsw_nb15_eda.png", dpi=110); plt.show()

/tmp/ipykernel_176493/3849618170.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/unsw_nb15_eda.png", dpi=110); plt.show()


In [9]:
numeric_summary(df, "unsw_nb15").head(15)

,count,mean,std,min,25%,50%,75%,max
dur,145222.0,1.385912e+00,5.323925e+00,0.0,0.018519,0.470197,1.012877,5.999999e+01
spkts,145222.0,3.179441e+01,1.785338e+02,1.0,8.000000,10.000000,22.000000,1.064600e+04
dpkts,145222.0,3.190848e+01,1.455831e+02,0.0,6.000000,8.000000,20.000000,1.101800e+04
sbytes,145222.0,1.443737e+04,2.295783e+05,24.0,564.000000,1008.000000,2766.000000,1.435577e+07
dbytes,145222.0,2.498134e+04,1.911670e+05,0.0,268.000000,678.000000,3276.000000,1.465753e+07
rate,145222.0,1.992546e+04,8.172167e+04,0.0,23.882346,57.744654,2742.230469,1.000000e+06
sload,145222.0,3.451662e+07,1.879390e+08,0.0,8019.374878,31365.746094,612318.437500,5.988000e+09
dload,145222.0,1.155375e+06,3.121509e+06,0.0,2858.841858,9309.610840,576354.062500,2.242273e+07
sloss,145222.0,8.376899e+00,8.652928e+01,0.0,2.000000,2.000000,6.000000,5.319000e+03
dloss,145222.0,1.165100e+01,7.005912e+01,0.0,1.000000,2.000000,6.000000,5.507000e+03


## Save clean + map to unified schema

In [10]:
save_clean(df, "unsw_nb15")
rep = dataset_report(df, "unsw_nb15", label_col="label",
                     notes="Curated parquet; no raw IP/timestamp; synthetic event_time flagged.")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/unsw_nb15.parquet (145,222 rows)
report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/unsw_nb15_stats.json


In [11]:
sev_map = {"normal": 0, "analysis": 2, "reconnaissance": 2, "fuzzers": 2,
           "dos": 3, "generic": 3, "exploits": 3,
           "backdoor": 4, "shellcode": 4, "worms": 4}
u = pd.DataFrame({
    "event_time": df["event_time"],
    "event_subtype": df["attack_cat"].astype(str),
    "protocol": df["proto"].astype(str),
    "duration_s": df["dur"],
    "bytes_out": df["sbytes"].astype("float64"),
    "bytes_in": df["dbytes"].astype("float64"),
    "severity": df["attack_cat"].astype(str).map(sev_map).astype("Int8"),
    "label": df["label"].astype("Int8"),
    "time_is_synthetic": True,
})
attr_cols = ["service", "state", "rate", "sload", "dload", "spkts", "dpkts", "tcprtt", "split"]
u[attr_cols] = df[attr_cols]
u = to_unified(u, source_dataset="unsw_nb15", event_domain="cyber",
               event_type="network_flow", label_type="attack", attributes_cols=attr_cols)
save_unified_part(u, "unsw_nb15")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_unsw_nb15.parquet (145,222 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,src_port,dst_port,protocol,country,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,unsw_nb15-0,2015-01-22 00:00:00.758508837+00:00,cyber,network_flow,normal,unsw_nb15,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,tcp,NaN,NaN,0.024525,3380.0,37268.0,0,0,attack,<NA>,True,"{""service"":""smtp"",""state"":""fin"",""rate"":3792.04..."
1,unsw_nb15-1,2015-01-22 00:00:00.765060779+00:00,cyber,network_flow,exploits,unsw_nb15,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,tcp,NaN,NaN,0.302579,514.0,18340.0,3,1,attack,<NA>,True,"{""service"":""unknown"",""state"":""fin"",""rate"":95.8..."
2,unsw_nb15-2,2015-01-22 00:00:01.108196154+00:00,cyber,network_flow,normal,unsw_nb15,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,tcp,NaN,NaN,0.045095,14990.0,7814.0,0,0,attack,<NA>,True,"{""service"":""unknown"",""state"":""fin"",""rate"":5477..."
